In [20]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd


BASE_DIR = Path.cwd()

MODEL_PATH = BASE_DIR / "models" / "isolation_forest.joblib"
FEATURES_PATH = BASE_DIR / "models" / "features.joblib"

THRESHOLD = 0.059


def load_model():
    """Load the trained model and its required feature list."""
    model = joblib.load(MODEL_PATH)
    features = joblib.load(FEATURES_PATH)
    return model, features


def predict_csv(file):
    """Validate a network-flow CSV and flag anomalous flows."""

    model, features = load_model()

    # Read uploaded CSV
    df = pd.read_csv(file)
    df.columns = df.columns.str.strip()

    # Validate required columns
    missing = [col for col in features if col not in df.columns]

    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    # Keep only the features used during training
    X = df[features].copy()

    # Convert values to numeric and handle invalid values
    for col in features:
        X[col] = pd.to_numeric(X[col], errors="coerce")

    X.replace([np.inf, -np.inf], np.nan, inplace=True)

    # The model's pipeline handles missing values using its imputer
    scores = model.decision_function(X)

    # Lower scores indicate more anomalous traffic
    anomaly = scores < THRESHOLD

    # Return results for the backend or frontend
    results = df.copy()
    results["anomaly_score"] = scores
    results["prediction"] = np.where(
        anomaly, "Anomaly", "Normal"
    )

    return results

In [8]:
import os

print("Current directory:", os.getcwd())
print("Files in data:", os.listdir("data"))

Current directory: /Users/himanshu/Documents/my documents/hackathon/cyvanta/ml
Files in data: ['Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv.zip']


In [9]:
import zipfile

with zipfile.ZipFile("data/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv.zip") as z:
    z.extractall("data")

print("Dataset extracted!")

Dataset extracted!


In [10]:
from pipeline import predict_csv

result = predict_csv("data/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv")

print(result["prediction"].value_counts())
result.head()

prediction
Normal     115497
Anomaly    110248
Name: count, dtype: int64


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,anomaly_score,prediction
0,54865,3,2,0,12,0,6,6,6.0,0.0,...,0.0,0,0,0.0,0.0,0,0,BENIGN,0.127690,Normal
1,55054,109,1,1,6,6,6,6,6.0,0.0,...,0.0,0,0,0.0,0.0,0,0,BENIGN,0.123464,Normal
2,55055,52,1,1,6,6,6,6,6.0,0.0,...,0.0,0,0,0.0,0.0,0,0,BENIGN,0.123464,Normal
3,46236,34,1,1,6,6,6,6,6.0,0.0,...,0.0,0,0,0.0,0.0,0,0,BENIGN,0.123464,Normal
4,54863,3,2,0,12,0,6,6,6.0,0.0,...,0.0,0,0,0.0,0.0,0,0,BENIGN,0.127690,Normal


In [11]:
from sklearn.metrics import classification_report, confusion_matrix

y_true = result["Label"].str.strip().map({
    "BENIGN": "Normal",
    "DDoS": "Anomaly"
})

print(confusion_matrix(y_true, result["prediction"]))
print(classification_report(y_true, result["prediction"]))

[[84776 43251]
 [25472 72246]]
              precision    recall  f1-score   support

     Anomaly       0.77      0.66      0.71    128027
      Normal       0.63      0.74      0.68     97718

    accuracy                           0.70    225745
   macro avg       0.70      0.70      0.69    225745
weighted avg       0.71      0.70      0.70    225745



In [14]:
import importlib
import pipeline

importlib.reload(pipeline)

output = pipeline.predict_csv(
    "data/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv"
)

print(output["summary"])
output["results"].head()

{'total_flows': 225745, 'anomaly_count': 110248, 'normal_count': 115497, 'anomaly_percentage': 48.84}


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,anomaly_score,prediction
0,54865,3,2,0,12,0,6,6,6.0,0.0,...,0.0,0,0,0.0,0.0,0,0,BENIGN,0.127690,Normal
1,55054,109,1,1,6,6,6,6,6.0,0.0,...,0.0,0,0,0.0,0.0,0,0,BENIGN,0.123464,Normal
2,55055,52,1,1,6,6,6,6,6.0,0.0,...,0.0,0,0,0.0,0.0,0,0,BENIGN,0.123464,Normal
3,46236,34,1,1,6,6,6,6,6.0,0.0,...,0.0,0,0,0.0,0.0,0,0,BENIGN,0.123464,Normal
4,54863,3,2,0,12,0,6,6,6.0,0.0,...,0.0,0,0,0.0,0.0,0,0,BENIGN,0.127690,Normal


In [15]:
import pandas as pd
from pipeline import predict_csv

# Create an invalid CSV missing the required features
pd.DataFrame({"WrongColumn": [1, 2, 3]}).to_csv(
    "data/invalid_test.csv", index=False
)

try:
    predict_csv("data/invalid_test.csv")
except ValueError as e:
    print("Expected validation error:", e)
    

Expected validation error: CSV is missing required network-flow features: Packet Length Std, Packet Length Mean, ACK Flag Count, Flow IAT Mean, Total Length of Bwd Packets, Total Fwd Packets, Flow Duration, Total Backward Packets


In [16]:
with open("data/empty_test.csv", "w") as f:
    pass

try:
    pipeline.predict_csv("data/empty_test.csv")
except ValueError as e:
    print(e)

The uploaded CSV is empty.


In [17]:
import pandas as pd

df = pd.DataFrame({
    feature: ["invalid", "invalid"]
    for feature in pipeline.load_model()[1]
})

df.to_csv("data/invalid_values_test.csv", index=False)

try:
    pipeline.predict_csv("data/invalid_values_test.csv")
except ValueError as e:
    print(e)

2 row(s) contain no usable values in the required features. Please check your CSV.


In [18]:
import pandas as pd

sample = pd.DataFrame({
    "Packet Length Std": [120.5, 0.0, 85.3, 200.1, 15.2],
    "Packet Length Mean": [250.0, 60.0, 180.5, 500.0, 75.0],
    "ACK Flag Count": [1, 0, 1, 1, 0],
    "Flow IAT Mean": [5000.0, 100.0, 2500.0, 800.0, 120.0],
    "Total Length of Bwd Packets": [1200, 0, 850, 5000, 50],
    "Total Fwd Packets": [10, 1, 8, 50, 2],
    "Flow Duration": [50000, 100, 25000, 80000, 500],
    "Total Backward Packets": [8, 0, 6, 40, 1]
})

sample.to_csv("data/sample_network_flows.csv", index=False)

print("Sample CSV created!")

Sample CSV created!


In [19]:
output = pipeline.predict_csv("data/sample_network_flows.csv")

print(output["summary"])
print(output["results"][["anomaly_score", "prediction"]])

{'total_flows': 5, 'anomaly_count': 3, 'normal_count': 2, 'anomaly_percentage': 60.0}
   anomaly_score prediction
0      -0.049332    Anomaly
1       0.121530     Normal
2      -0.024880    Anomaly
3      -0.157611    Anomaly
4       0.156757     Normal


In [ ]:
print(df.columns.tolist())
print(df.shape)